In [ ]:
# 3. GPT-2 test-drive

In [ ]:
!pip install accelerate -U
!pip install transformers[torch]

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    LineByLineTextDataset,
    TextDataset,
    TrainingArguments,
    Trainer,
)

In [ ]:
batch_size = 8
epochs = 1

train_data_file = "../data/interim/combined_text.txt"
model_output_dir = "../models/gpt2-based"

warmup_steps = 100
save_steps = 100
logging_steps = 100

In [ ]:
# Make a smaller file of 2000 lines for testing
# !head -2000 ../data/interim/combined_text.txt > ../data/interim/combined_text_2000.txt

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2", cache_dir="cache")
tokenizer = AutoTokenizer.from_pretrained("gpt2", cache_dir="cache")
datacollator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
train_dataset = LineByLineTextDataset(
    tokenizer=tokenizer,
    file_path=train_data_file,
    block_size=64,
)

In [ ]:
training_args = TrainingArguments(
    output_dir=model_output_dir,
    overwrite_output_dir=True,
    num_train_epochs=epochs,
    per_gpu_train_batch_size=batch_size,
    warmup_steps=warmup_steps,
    save_steps=save_steps,
    logging_steps=logging_steps,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=datacollator,
    train_dataset=train_dataset,
)

trainer.train()
trainer.save_model(model_output_dir)

In [ ]:
# Inference
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device="cuda")

In [ ]:
def detoxify(text):
    return generator(f"[s]{text}[/s]»[t]")

In [ ]:
prompt = "I hate your stupid fucking face!!"

detoxify(prompt)